In [ ]:
import kagglehub


# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
#importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Write your code here:
df_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(df_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])
df.head()

In [ ]:
# Task 2: Write your code here:
print(df.isnull().sum())

In [ ]:
df.tail(20)
df_example = df.copy()
df_example['Delivery_Time'] = df['Delivery_Time'].fillna(-1)
df_example.head()

In [ ]:
df_example =  df_example[df_example['Delivery_Time']<0]
df_example.head()

In [ ]:
#since Delivery Time is our target , we will drop the columns with Nan values overthere(can't predict NaN) , also NaN rows have normal values in the rest columns
df = df.dropna(subset = ['Delivery_Time'])

In [ ]:
df.isnull().sum()

In [ ]:
df['Weather'] = df['Weather'].fillna("Unknown")
df = df.dropna(subset=['Traffic_Level'])
df = df.dropna(subset=['Time_of_Day'])
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(0)
df.info()


In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
categorical_cols = ['Weather', 'Traffic_Level','Time_of_Day','Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

In [ ]:
# Task 5: Write your code here:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()
#some outliers but they dont hurt

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score


n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

lr_losses = []
lr_mae = []


for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
  model.fit(X_train, y_train)
  print("Model trained!")

  # Validate
  y_pred = model.predict(X_test)



  # Calculate evaluation metrics
  mae = mean_absolute_error(y_test, y_pred)


  # Store results
  lr_mae.append(mae)

average_mae = np.mean(lr_mae, axis=0)
print(average_mae)

In [ ]:
df.info()

In [ ]:
# Task 1: Write your code here:
feature_cols = ['Distance_km' , 'Weather' , 'Traffic_Level', 'Time_of_Day' , 'Vehicle_Type' ,'Preparation_Time_min' ,'Courier_Experience_yrs' ]
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Scatter plot: Predicted vs Actual
plt.figure(figsize=(8, 6))

# Plot: Predicted vs Actual scatter
plt.scatter(y_test, y_pred.flatten(), alpha=0.5, s=10, c='steelblue')

# Add perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')

plt.xlabel('Actual Price ($)', fontsize=12)
plt.ylabel('Predicted Price ($)', fontsize=12)
plt.title('Predicted vs Actual Diamond Prices', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: